# 1.FastIO

In [ ]:
from io import BytesIO, IOBase
import sys
import os

# import time
import bisect
# import functools
import math
import random
# import re
from collections import Counter, defaultdict, deque
# from copy import deepcopy
from functools import cmp_to_key, lru_cache, reduce
from heapq import heapify, heappop, heappush, heappushpop, nlargest, nsmallest
from itertools import accumulate, combinations, permutations
# from operator import add, iand, ior, itemgetter, mul, xor
# from string import ascii_lowercase, ascii_uppercase
from typing import *

input = lambda: sys.stdin.readline().rstrip("\r\n")


def I():
    return input()


def II():
    return int(input())


def MII():
    return map(int, input().split())


def LI():
    return list(input().split())


def LII():
    return list(map(int, input().split()))


def GMI():
    return map(lambda x: int(x) - 1, input().split())


def LGMI():
    return list(map(lambda x: int(x) - 1, input().split()))

sys.setrecursionlimit(int(1e5 + 10))
dx, dy = [0, 1, 0, -1, 1, -1, 1, -1], [1, 0, -1, 0, -1, -1, 1, 1]
inf = float('inf')

# 2.栈装饰器

In [ ]:
# --------------------
# 手写栈模板
# 克服py栈太浅的问题
from types import GeneratorType


def bootstrap(f, stack=[]):
    def wrappedfunc(*args, **kwargs):
        if stack:
            return f(*args, **kwargs)
        else:
            to = f(*args, **kwargs)
            while True:
                if type(to) is GeneratorType:
                    stack.append(to)
                    to = next(to)
                else:
                    stack.pop()
                    if not stack:
                        break
                    to = stack[-1].send(to)
            return to

    return wrappedfunc
# --------------------


# 3.点到其他点距离和

In [ ]:
from itertools import accumulate
from typing import List


def getSumAbsoluteDifferences(self, nums: List[int]) -> List[int]:
    n = len(nums)
    s = list(accumulate(nums))
    res = [0] * n          # res[i] = 前面的个数 * cur - 前面总和 + 后面的和 - 后面个数 * cur
    for i, x in enumerate(nums):
        res[i] = (i + 1) * x - s[i] + (s[n - 1] - s[i]) - (n - i - 1) * x
    return res


# 4.二维前缀和

In [ ]:
n, m, q = map(int, input().split())
N = 1010
a = [[0]]
s = [[0] * N for _ in range(N)]
for _ in range(n):
    a.append([0] + list(map(int, input().split())))
for i in range(1, n + 1):
    for j in range(1, m + 1):
        s[i][j] = s[i - 1][j] + s[i][j - 1] + a[i][j] - s[i - 1][j - 1]
for _ in range(q):
    x1, y1, x2, y2 = list(map(int,input().split()))
    print(s[x2][y2] + s[x1 - 1][y1 - 1] - s[x1 - 1][y2] - s[x2][y1 - 1])  # (x1, y1) (x2, y2) 的所有元素


# 下标从0开始的做法
"""
s = [[0] * (m + 1) for _ in range(n + 1)]
        for i in range(n):
            for j in range(m):
                s[i + 1][j + 1] = s[i + 1][j] + s[i][j + 1] - s[i][j] + (pizza[i][j] == 'A')
def query(x1, y1, x2, y2):
    print(s[x2][y2] + s[x1 - 1][y1 - 1] - s[x1 - 1][y2] - s[x2][y1 - 1])   # 这里和上面一样
"""

# 超大的子矩阵 https://atcoder.jp/contests/abc331/tasks/abc331_d

"""
n, q = MII()
g = [I() for _ in range(n)]
a = [[0] * n for _ in range(n)]
for i in range(n):
    for j in range(n):
        a[i][j] = 1 if g[i][j] == 'B' else 0
g.clear()
s = [[0] * (n + 1) for _ in range(n + 1)]
for i in range(n):
    for j in range(n):
        s[i + 1][j + 1] = s[i + 1][j] + s[i][j + 1] - s[i][j] + a[i][j]


def calc(r, c):
    r += 1
    c += 1
    nr, r = divmod(r, n)
    nc, c = divmod(c, n)
    return s[n][n] * nr * nc + nr * s[n][c] + nc * s[r][n] + s[r][c]

def query(x1, y1, x2, y2):
    x1 -= 1
    y1 -= 1
    return calc(x2, y2) - calc(x2, y1) - calc(x1, y2) + calc(x1, y1)

for _ in range(q):
    x1, y1, x2, y2 = MII()
    res = query(x1, y1, x2, y2)
    print(res)

"""

# 5.矩阵填充转向

In [ ]:
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def spiralMatrix(self, m: int, n: int, head):
        DIRS = ((0, 1), (1, 0), (0, -1), (-1, 0))  # 右  下  左 上
        res = [[-1] * n for _ in range(m)]
        i = j = di = 0
        while head:
            res[i][j] = head.val
            head = head.next
            dx, dy = DIRS[di]
            if (not 0 <= i + dx < m) or (not 0 <= j + dy < n) or res[i + dx][j + dy] != -1:  # 如果越界了更换方向
                di = (di + 1) % 4
            dx, dy = DIRS[di]
            i += dx
            j += dy
        return res


# 6.大小为K的子集

In [ ]:
# LC2397. 被列覆盖的最多行数
# 1.直接枚举所有情况
def maximumRows(mat, cols) -> int:
    mask = [sum(v << j for j, v in enumerate(row)) for row in mat]
    res = 0
    for set in range(1 << len(mat[0])):
        if set.bit_count() != cols:
            continue
        cnt = 0
        for row in mask:
            if row & set == row:
                cnt += 1
        res = max(res, cnt)
    return res


# 2.下一个排列算法

# lowbit，取到最低位的1
def lowbit(x):
    return x & -x


# x = 0101110 的下一个：找到第一个01翻转，并且把后面的1都放到末尾
# y = 0110011
# 左半部分 = x + lb
# 右半部分 = （x ^ (x + lb)）// lb >> 2
# 时间复杂度O(C(n, cols) * m)
def maximumRows(mat, cols) -> int:
    mask = [sum(v << j for j, v in enumerate(row)) for row in mat]
    res = 0
    x = (1 << cols) - 1
    while x < (1 << len(mat[0])):
        res = max(res, sum(row & x == row for row in mask))
        lb = x & -x
        left = x + lb
        right = (x ^ (x + lb)) // lb >> 2
        x = left | right
    return res


# n个里面选k个
def combine(n, k):
    res = []
    x = (1 << k) - 1
    while x < (1 << n):
        res.append([i + 1 for i in range(x.bit_length()) if x >> i & 1])
        lb = x & -x
        left = x + lb
        right = (x ^ (x + lb)) // lb >> 2
        x = left | right
    return res
